# JSON Schema 使用

除了 Pydantic 模型和 `TypedDict`，`with_structured_output` 还支持直接传入一个符合 [JSON Schema](https://json-schema.org/) 规范的 `dict`。这种方式最贴近模型底层的 function calling 协议，跨语言通用，适合需要动态拼装 schema 或在配置里描述结构的场景。

In [1]:
import json
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 与其他示例保持一致：关闭思考模式，避免结构化输出时 tool_choice 不被支持
model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


## 基础用法

In [2]:
## 基础用法：直接传入 JSON Schema 字典

person_schema = {
    "title": "Person",              # schema 名称
    "description": "人物信息",
    "type": "object",               # 顶层必须是 object
    "properties": {                 # 各字段的定义
        "name": {"type": "string", "description": "姓名"},
        "age": {"type": "integer", "description": "年龄"},
        "occupation": {"type": "string", "description": "职位"},
    },
    "required": ["name", "age", "occupation"],   # 必填字段
}

model_person = model.with_structured_output(schema=person_schema)
response = model_person.invoke("小许是一个28岁的Java开发工程师")

print(response)
print("返回类型：", type(response))   # 返回的是普通 dict


{'name': '小许', 'age': 28, 'occupation': 'Java开发工程师'}
返回类型： <class 'dict'>


## 必填与可选字段

In [3]:
## required 决定哪些字段必填；没写进 required 的字段即为可选

contact_schema = {
    "title": "Contact",
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "姓名"},
        "phone": {"type": "string", "description": "电话"},
        "email": {"type": "string", "description": "邮箱"},
    },
    "required": ["name"],   # 只有 name 必填，phone / email 可选
}

model_contact = model.with_structured_output(schema=contact_schema)
print(model_contact.invoke("小许，电话 13812345678"))


{'name': '小许', 'phone': '13812345678'}


## 枚举 enum

In [4]:
## enum 限定字段只能取固定值

resume_schema = {
    "title": "Resume",
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "姓名"},
        "level": {
            "type": "string",
            "enum": ["初级", "中级", "高级", "专家"],   # 只能取其中之一
            "description": "职级",
        },
        "city": {"type": "string", "description": "城市"},
    },
    "required": ["name", "level", "city"],
}

model_resume = model.with_structured_output(schema=resume_schema)
print(model_resume.invoke("张三是高级Java开发工程师，base 在上海"))


{'name': '张三', 'level': '高级', 'city': '上海'}


## 数组 array

In [5]:
## type=array + items 表示列表，items 描述每个元素的类型

movie_schema = {
    "title": "Movie",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影标题"},
        "actors": {
            "type": "array",
            "items": {"type": "string"},     # 字符串列表
            "description": "主演列表",
        },
        "genres": {
            "type": "array",
            "items": {"type": "string"},
            "description": "类型标签列表",
        },
    },
    "required": ["title", "actors", "genres"],
}

model_movie = model.with_structured_output(schema=movie_schema)
print(model_movie.invoke("请介绍电影《流浪地球》的主演和类型"))


{'title': '流浪地球', 'actors': ['吴京', '屈楚萧', '李光洁', '吴孟达', '赵今麦'], 'genres': ['科幻', '灾难', '冒险']}


## 嵌套对象

In [6]:
## properties 里再放一个 type=object，就形成嵌套结构

employee_schema = {
    "title": "Employee",
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "姓名"},
        "address": {
            "type": "object",
            "description": "住址",
            "properties": {
                "city": {"type": "string", "description": "城市"},
                "street": {"type": "string", "description": "街道"},
            },
            "required": ["city", "street"],
        },
    },
    "required": ["name", "address"],
}

model_employee = model.with_structured_output(schema=employee_schema)
print(model_employee.invoke("小许住在北京市朝阳区望京街道"))


{'name': '小许', 'address': {'city': '北京市', 'street': '朝阳区望京街道'}}


## $defs / $ref 复用子 schema

In [7]:
## 用 $defs 定义可复用的子 schema，再用 $ref 引用，避免重复书写

order_schema = {
    "title": "Order",
    "type": "object",
    "properties": {
        "orderId": {"type": "string", "description": "订单号"},
        "customer": {"$ref": "#/$defs/Customer"},          # 引用下面的定义
        "shipping": {"$ref": "#/$defs/Address"},
    },
    "required": ["orderId", "customer", "shipping"],
    "$defs": {                                             # 可复用的子定义
        "Address": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "street": {"type": "string"},
            },
            "required": ["city", "street"],
        },
        "Customer": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "address": {"$ref": "#/$defs/Address"},
            },
            "required": ["name", "address"],
        },
    },
}

model_order = model.with_structured_output(schema=order_schema)
print(model_order.invoke("订单 A1001，客户小许，收货地址是北京市朝阳区望京街道"))


{'orderId': 'A1001', 'customer': {'name': '小许', 'address': {'city': '北京市', 'street': '朝阳区望京街道'}}, 'shipping': {'city': '北京市', 'street': '朝阳区望京街道'}}


## 数值与字符串约束

In [8]:
## 用 JSON Schema 的关键字为字段加取值范围约束

product_schema = {
    "title": "Product",
    "type": "object",
    "properties": {
        "name": {
            "type": "string",
            "minLength": 2,       # 最短 2 个字符
            "maxLength": 20,      # 最长 20 个字符
            "description": "商品名称",
        },
        "price": {
            "type": "number",
            "exclusiveMinimum": 0,   # 必须大于 0
            "description": "价格",
        },
        "stock": {
            "type": "integer",
            "minimum": 0,            # 必须大于等于 0
            "description": "库存",
        },
        "sku": {
            "type": "string",
            "pattern": "^[A-Z]{2}-\\d+$",   # 正则约束，如 AB-123
            "description": "商品编码",
        },
    },
    "required": ["name", "price", "stock"],
}

model_product = model.with_structured_output(schema=product_schema)
print(model_product.invoke("一款名为机械键盘的商品，售价 399.5，库存 12，编码 KB-123"))


{'name': '机械键盘', 'price': 399.5, 'stock': 12, 'sku': 'KB-123'}


## additionalProperties 禁止额外字段

In [9]:
## additionalProperties=False 时，模型不允许输出 schema 里没有定义的字段

strict_schema = {
    "title": "StrictUser",
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "姓名"},
        "age": {"type": "integer", "description": "年龄"},
    },
    "required": ["name", "age"],
    "additionalProperties": False,   # 只允许上面两个字段
}

model_strict = model.with_structured_output(schema=strict_schema)
print(model_strict.invoke("小许今年 28 岁，喜欢编程和篮球"))   # 爱好不会被输出


{'name': '小许', 'age': 28}


## 与工具结合

In [10]:
## JSON Schema 也可以作为工具的 args_schema
from langchain_core.tools import tool

weather_schema = {
    "type": "object",
    "properties": {
        "city": {"type": "string", "description": "城市名称，如北京、上海"},
    },
    "required": ["city"],
}

@tool(args_schema=weather_schema)
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city}今天晴，25℃"

print(get_weather.invoke({"city": "北京"}))

## 也可以直接查看工具转成的 JSON Schema
from langchain_core.utils.function_calling import convert_to_openai_tool
print(json.dumps(convert_to_openai_tool(get_weather), ensure_ascii=False, indent=2))


北京今天晴，25℃
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "查询指定城市的天气",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "城市名称，如北京、上海"
        }
      },
      "required": [
        "city"
      ]
    }
  }
}


## 三种 schema 定义方式对比

| 方式 | 运行时校验 | 默认值/别名 | 返回类型 | 适用场景 |
| --- | --- | --- | --- | --- |
| JSON Schema `dict` | 无（取决于服务端） | 不支持 | `dict` | 动态拼装、跨语言、最贴近底层协议 |
| `TypedDict` | 无 | 不支持 | `dict` | 结构简单、想要类型提示 |
| Pydantic | 有 | 支持 | 模型实例 | 需要严格校验、复杂逻辑 |

三者最终都会被转换成 JSON Schema 交给模型，选择哪种取决于你对类型与校验的需求。